# Score notes as a bopp extension

Brian McFee's PR #8 (bmcfee/bopp) adds `ext` payloads, whose items are validated in a second pass by a type that an installed package registers under the `bopp_extension` entry-point group. The `bopp-score-ext` plugin in `experiments/plugin/` registers `io.github.johentsch.score_note:v1` as a `ScoreNote` struct that mirrors the columnar `score_note` payload. This notebook builds a `score_note` annotation from a Distant Listening Corpus note table, re-encodes it as an `ext` annotation, and shows the ids, file sizes, the no-plugin fallback, validation errors and timings.

In [1]:
import tempfile
import timeit
import warnings
from pathlib import Path

import msgspec
from bopp_score_ext import EXT_SCHEMA, from_ext, to_ext

import bopp
import bopp.core
import bopp.extensions
import bopp.util
from bopp.io import load_bopp_msgpack, save_bopp_json, save_bopp_msgpack
from experiments.dlc_notes import notes_to_score_note, read_notes_tsv

# bopp is imported from the repository checkout, so the fixture is located from the package,
# not from the kernel working directory (the notebook may be run from a copy elsewhere)
REPO = Path(bopp.__file__).resolve().parents[1]
FIXTURE = REPO / "tests" / "experiments" / "data" / "dlc_notes_chopin_mazurkas_BI163op67-4.tsv"
WORK = Path(tempfile.mkdtemp())

ann = notes_to_score_note(read_notes_tsv(FIXTURE))
print("notes:", len(ann.payload.midi))
print("extent:", type(ann.extent).__name__)
print(bopp.util.to_dataframe(ann).head(3).to_string())

notes: 822
extent: QuartersIntervalFraction
   payload:score_note:midi  payload:score_note:tpc payload:score_note:name  payload:score_note:octave  payload:score_note:staff  payload:score_note:voice  payload:score_note:mc  payload:score_note:mn  payload:score_note:tied payload:score_note:gracenote payload:score_note:nominal_duration payload:score_note:scalar  payload:score_note:chord_id  payload:score_note:volta extent:quarters_interval.fraction:quarter extent:quarters_interval.fraction:duration
0                       76                       4                      E5                          5                         1                         1                      1                      0                      1.0                          NaN                              [1, 4]                    [1, 1]                            0                       NaN                                    [0, 1]                                     [1, 1]
1                       45                  

In [2]:
ext = to_ext(ann)
print("ext_schema:", ext.payload.ext_schema)
print("first row type:", type(ext.payload.value[0]).__name__)
print("first row:", msgspec.to_builtins(ext.payload.value[0]))
print("columnar id:", ann.id)
print("ext id:", ext.id)

ext_schema: io.github.johentsch.score_note:v1
first row type: ScoreNote
first row: {'midi': 76, 'tpc': 4, 'name': 'E5', 'octave': 5, 'staff': 1, 'voice': 1, 'mc': 1, 'mn': 0, 'tied': 1, 'nominal_duration': [1, 4], 'scalar': [1, 1], 'chord_id': 0}
columnar id: 9e09e71c-b72a-5ea7-af07-430412d1d877
ext id: 0708a693-7d74-5267-abe0-7c04fa7ea9d9


In [3]:
save_bopp_json(ann, WORK / "columnar.json")
save_bopp_msgpack(ann, WORK / "columnar.msgpack")
save_bopp_json(ext, WORK / "ext.json")
save_bopp_msgpack(ext, WORK / "ext.msgpack")
print(f"{'payload':<10}{'json (bytes)':>14}{'msgpack (bytes)':>17}")
for form in ("columnar", "ext"):
    json_size = (WORK / f"{form}.json").stat().st_size
    msgpack_size = (WORK / f"{form}.msgpack").stat().st_size
    print(f"{form:<10}{json_size:>14}{msgpack_size:>17}")

payload     json (bytes)  msgpack (bytes)
columnar           54919            23490
ext               122046            82927


In [4]:
resolved = load_bopp_msgpack(WORK / "ext.msgpack")
unresolved = load_bopp_msgpack(WORK / "ext.msgpack", resolve_ext=False)
print("first row, resolved:", type(resolved.payload.value[0]).__name__)
print("first row, unresolved:", type(unresolved.payload.value[0]).__name__)
ids_match = ext.id == resolved.id == unresolved.id
both_valid = bopp.core.validate_annotation_id(resolved) and bopp.core.validate_annotation_id(unresolved)
print("ids equal (plugin vs no plugin):", ids_match and both_valid)

first row, resolved: ScoreNote
first row, unresolved: dict
ids equal (plugin vs no plugin): True


In [5]:
bopp.extensions.REGISTRY.clear()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    without_plugin = load_bopp_msgpack(WORK / "ext.msgpack")
print("warning:", next(str(w.message) for w in caught if "No extension registered" in str(w.message)))
print("row type without plugin:", type(without_plugin.payload.value[0]).__name__)
print("id unchanged:", without_plugin.id == ext.id)
bopp.extensions.reset_extensions()
print("schema registered again:", EXT_SCHEMA in bopp.extensions.get_extensions())

row type without plugin: dict
id unchanged: True
schema registered again: True


In [6]:
for row in ({"midi": 128, "tpc": 0}, {"midi": 60, "tpc": 0, "unknown": 1}):
    try:
        bopp.create(media_id="demo:bad", payload_kind="ext", ext_schema=EXT_SCHEMA, value=[row])
    except msgspec.ValidationError as err:
        print(type(err).__name__, "-", err)
    else:
        print("accepted (unexpected)")

ValidationError - Expected `int` <= 127 - at `$[0].midi`
ValidationError - Object contains unknown field `unknown` - at `$[0]`


In [7]:
print("from_ext(to_ext(ann)).id == ann.id:", from_ext(to_ext(ann)).id == ann.id)

from_ext(to_ext(ann)).id == ann.id: True


In [8]:
def best_ms(stmt):
    """Best of 5 repeats of 20 calls, per call, in milliseconds."""
    return min(timeit.repeat(stmt, number=20, repeat=5)) / 20 * 1000


print(f"columnar msgpack decode: {best_ms(lambda: load_bopp_msgpack(WORK / 'columnar.msgpack')):.3f} ms")
print(f"ext msgpack, first pass only (resolve_ext=False): {best_ms(lambda: load_bopp_msgpack(WORK / 'ext.msgpack', resolve_ext=False)):.3f} ms")
print(f"ext msgpack, two passes (resolve_ext=True): {best_ms(lambda: load_bopp_msgpack(WORK / 'ext.msgpack')):.3f} ms")

columnar msgpack decode: 2.053 ms
ext msgpack, first pass only (resolve_ext=False): 3.639 ms
ext msgpack, two passes (resolve_ext=True): 2.940 ms


The `ext` form has the same id with the plugin resolved, with rows left as dicts, and after a JSON or MsgPack round trip, because the rows serialize to the same bytes either way. Row validation is the second pass: it runs only when the plugin is registered, so a file stays readable without it and its id still checks out. In this run the second pass (row validation) added about 0.4 ms to the first pass, while the `ext` file is larger and slower to decode than the columnar one, so most of the extra cost comes from the row-oriented first pass. The exact numbers vary between runs on this machine. Two caveats from the plugin README remain: CSV round trips of `ext` payloads are not id-stable upstream yet, and a column that is entirely `None` does not survive `to_ext`.